# Indo-1 kinetics analysis -- fixed for Oscar's Oct 2026 folders

Same method as `Ca_flow_analysis_mean_se.ipynb`. Three things changed:

1. **File discovery.** The original hard-codes names like `export_liveNK92_344663_NK92_zeta.fcs`.
   Oscar's actual exports are `export_liveNK92_344663_NK92 (CD3z).fcs` (Exp 1) and
   `export_20260618NK_361774_NK92 (CD3z).fcs` (Exp 2). This version finds the four tubes
   by the tag in parentheses, so it runs on either folder unmodified.
2. **Time-scale assertion.** Added a check against `$PnR`, the declared range of the Time
   parameter. FlowKit's `get_events(source="raw")` applies the `$PnG` gain (0.01); the
   notebook then applies `$TIMESTEP` (0.01). Net 1e-4, giving 331 s, which matches
   `$P7R = 331`. If a future FlowKit version stops applying gain, the time axis silently
   becomes 33133 s and every fit is garbage. The assertion catches that.
3. **Empty-bin handling.** In Experiment 2 the CD3z and FcRg tubes stop at ~222 s while
   CD3z+FcRg runs to 329 s. The original puts 250 bins across the global max, so the last
   ~80 rows come out NaN for zeta and gamma. `TRIM = True` drops them.

Everything else -- 250 bins, `np.digitize`, `std(ddof=1)`, `SE = std/sqrt(n)`, and the
exported column set -- is unchanged.

In [ ]:
# Cell 1: imports, folder, settings

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import flowkit

# Point this at ONE experiment folder (the one holding the four .fcs files).
data_dir = Path.cwd()

n_bins = 250      # unchanged from the original notebook
TRIM   = True     # drop bins where any of zeta/gamma/hetero has no events
OUT    = data_dir / "Ca_NK92.csv"

# Oscar labels the tube in the trailing parentheses of the file name.
# Both of his naming conventions are covered.
TAG_TO_LABEL = {
    "nk92": "NK92",        "parental": "NK92",
    "cd3z": "zeta",        "ki-1 cd3z_cd16": "zeta",
    "fcrg": "gamma",       "ki-2 fcg_cd16": "gamma",
    "cd3z+fcrg": "hetero", "ki-6 fcg_cd3z_cd16": "hetero",
}

def tube_tag(path):
    name = path.name
    if "(" in name and ")" in name:
        return name[name.rfind("(") + 1:name.rfind(")")].strip().lower()
    return path.stem.strip().lower()

fcs_paths = sorted(p for p in data_dir.rglob("*.fcs")
                   if "__MACOSX" not in p.parts and not p.name.startswith("._"))

file_names = {}
for p in fcs_paths:
    tag = tube_tag(p)
    label = TAG_TO_LABEL.get(tag) or next(
        (v for k, v in TAG_TO_LABEL.items() if k in tag), None)
    if label is None:
        print(f"skipping (unrecognised tube tag {tag!r}): {p.name}")
        continue
    if label in file_names:
        raise ValueError(f"two files map to {label}: {file_names[label]} and {p.name}")
    file_names[label] = p.name

missing = [k for k in ("zeta", "gamma", "hetero") if k not in file_names]
if missing:
    raise ValueError(f"missing tubes {missing} in {data_dir}")

for k in ("NK92", "zeta", "gamma", "hetero"):
    if k in file_names:
        print(f"{k:>7} <- {file_names[k]}")

samples = {
    label: flowkit.Sample(str(data_dir / file_name))
    for label, file_name in file_names.items()
}

In [ ]:
# Cell 2: time conversion, with the $PnR assertion

def get_timestep_seconds(sample):
    metadata = sample.get_metadata()
    timestep_raw = metadata.get("$TIMESTEP", metadata.get("$timestep", metadata.get("timestep")))
    if timestep_raw is None:
        return None
    try:
        timestep = float(timestep_raw)
    except (TypeError, ValueError):
        return None
    return timestep if timestep > 0 else None


def raw_time_to_seconds(raw_time, sample):
    timestep = get_timestep_seconds(sample)
    if timestep is None:
        # If TIMESTEP is missing, assume Time is already in seconds.
        return raw_time
    return raw_time * timestep


def parameter_names(sample):
    return sample.channels["pnn"].fillna("").astype(str).tolist()


def find_channel(sample, patterns, description):
    names = parameter_names(sample)
    matches = [n for n in names if any(p in n.lower() for p in patterns)]
    if not matches:
        raise ValueError(f"Could not find {description}. Channels: {names}")
    return matches[0]


def declared_time_range(sample):
    """$PnR for the Time parameter -- the acquisition duration in seconds."""
    names = parameter_names(sample)
    time_name = find_channel(sample, ["time", "timestamp"], "Time channel")
    n = names.index(time_name) + 1
    md = sample.get_metadata()
    for key in (f"$p{n}r", f"$P{n}R", f"p{n}r"):
        if key in md:
            try:
                return float(md[key])
            except (TypeError, ValueError):
                pass
    return None


def check_time_scale(label, sample, time_seconds):
    """FlowKit should already have applied the $PnG gain inside source='raw'.
    If it has, converted max ~= $PnR. If it has not, we are off by 1/gain = 100x."""
    r = declared_time_range(sample)
    tmax = float(np.nanmax(time_seconds))
    if r is None:
        print(f"  {label:>7}: tmax = {tmax:8.1f} s  ($PnR not declared, cannot check)")
        return
    ok = 0.5 * r <= tmax <= 2.0 * r
    print(f"  {label:>7}: tmax = {tmax:8.1f} s  vs $PnR = {r:6.0f}   {'OK' if ok else 'MISMATCH'}")
    if not ok:
        raise AssertionError(
            f"{label}: converted time max {tmax:.1f} s disagrees with declared range {r:.0f} s. "
            f"Most likely flowkit's get_events(source='raw') is no longer applying the $PnG "
            f"gain. Multiply by the gain explicitly before continuing."
        )


print("time-scale check")
for label, sample in samples.items():
    names = parameter_names(sample)
    events = sample.get_events(source="raw")
    time_name = find_channel(sample, ["time", "timestamp"], "Time channel")
    t = raw_time_to_seconds(events[:, names.index(time_name)], sample)
    check_time_scale(label, sample, t)

In [ ]:
# Cell 3: extract Time and Indo-1 ratio (unchanged)

raw_data = {}

for label, sample in samples.items():
    names = parameter_names(sample)
    events = sample.get_events(source="raw")
    name_to_index = {name: i for i, name in enumerate(names)}

    time_name = find_channel(sample, ["time", "timestamp"], "Time channel")
    ratio_name = find_channel(sample, ["indo-1 high_low ratio", "indo", "ratio"],
                              "Indo-1 ratio channel")

    raw_time = events[:, name_to_index[time_name]]
    time_seconds = raw_time_to_seconds(raw_time, sample)
    indo_ratio = events[:, name_to_index[ratio_name]]

    valid = np.isfinite(time_seconds) & np.isfinite(indo_ratio)
    raw_data[label] = pd.DataFrame({
        "time_seconds": time_seconds[valid],
        "indo_ratio": indo_ratio[valid],
    })

for label, df in raw_data.items():
    print(f"{label:>7}: {len(df):,} events, "
          f"time = {df['time_seconds'].min():.3f} to {df['time_seconds'].max():.3f} s")

In [ ]:
# Cell 4: bin, mean +/- SE, export (binning logic unchanged; NaN handling added)

global_max_time = max(df["time_seconds"].max() for df in raw_data.values())
bin_edges = np.linspace(0, global_max_time, n_bins + 1)
time_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

summary = pd.DataFrame({"time_seconds": time_centers})

for label, df in raw_data.items():
    bin_id = np.digitize(df["time_seconds"], bin_edges) - 1
    valid_bin = (bin_id >= 0) & (bin_id < n_bins)

    temp = pd.DataFrame({
        "bin": bin_id[valid_bin],
        "indo_ratio": df.loc[valid_bin, "indo_ratio"].to_numpy(),
    })
    grouped = temp.groupby("bin")["indo_ratio"]

    mean_values = grouped.mean().reindex(range(n_bins))
    std_values = grouped.std(ddof=1).reindex(range(n_bins))
    n_values = grouped.count().reindex(range(n_bins), fill_value=0)
    se_values = std_values / np.sqrt(n_values.replace(0, np.nan))

    summary[f"mean_{label}"] = mean_values.to_numpy()
    summary[f"SE_{label}"] = se_values.to_numpy()
    summary[f"n_{label}"] = n_values.to_numpy()

# --- report empty bins; in Exp 2 the zeta and gamma tubes stop ~100 s early
print(f"bin width = {global_max_time / n_bins:.4f} s over 0..{global_max_time:.2f} s\n")
for label in ("zeta", "gamma", "hetero"):
    bad = summary[f"mean_{label}"].isna()
    if bad.any():
        print(f"  {label:>7}: {bad.sum()} empty bins, "
              f"{summary.loc[bad, 'time_seconds'].min():.1f}-"
              f"{summary.loc[bad, 'time_seconds'].max():.1f} s "
              f"(tube ends at {raw_data[label]['time_seconds'].max():.1f} s)")

export_cols = ["time_seconds",
               "mean_zeta", "mean_gamma", "mean_hetero",
               "SE_zeta", "SE_gamma", "SE_hetero"]
export_table = summary[export_cols]

if TRIM:
    before = len(export_table)
    keep = ~export_table[["mean_zeta", "mean_gamma", "mean_hetero"]].isna().any(axis=1)
    export_table = export_table[keep].reset_index(drop=True)
    print(f"\nTRIM: dropped {before - len(export_table)} bins; kept "
          f"{export_table.time_seconds.min():.2f}-{export_table.time_seconds.max():.2f} s")

export_table.to_csv(OUT, index=False)
print(f"\nSaved: {OUT}   {len(export_table)} rows")
export_table.head()

In [ ]:
# Cell 5: plots (unchanged)

colors = {"NK92": "black", "zeta": "blue", "gamma": "red", "hetero": "cyan"}
order = [k for k in ("NK92", "zeta", "gamma", "hetero") if k in raw_data]

fig, ax = plt.subplots(figsize=(7, 5), constrained_layout=True)
for label in order:
    df = raw_data[label]
    ax.scatter(df["time_seconds"], df["indo_ratio"], s=2, alpha=0.15,
               color=colors[label], label=label, rasterized=True)
ax.set_title("Raw Indo-1 Ratio vs Time")
ax.set_xlabel("Time (seconds)")
ax.set_ylabel("Indo-1 high_low Ratio")
ax.legend(); ax.grid(alpha=0.25)
fig.savefig("ca_raw.png", dpi=150)
plt.show()

fig, ax = plt.subplots(figsize=(7, 5), constrained_layout=True)
for label in order:
    m, s = summary[f"mean_{label}"], summary[f"SE_{label}"]
    ax.plot(summary["time_seconds"], m, color=colors[label], linewidth=1.5, label=label)
    ax.fill_between(summary["time_seconds"], m - s, m + s, color=colors[label], alpha=0.15)
ax.set_title("Mean Indo-1 Ratio +/- SE")
ax.set_xlabel("Time (seconds)")
ax.set_ylabel("Mean Indo-1 high_low Ratio")
ax.legend(); ax.grid(alpha=0.25)
fig.savefig("ca_mean_se.png", dpi=150)
plt.show()

print("wrote ca_raw.png and ca_mean_se.png -- view them over python -m http.server")

In [ ]:
# Cell 6: cross-check against parse_ca_fcs.py (no flowkit, independent FCS reader)
#
#   python3 ~/nk92-ca-data/parse_ca_fcs.py . -o /tmp/cli.csv
#   python3 ~/nk92-ca-data/compare_ca_csv.py Ca_NK92.csv /tmp/cli.csv
#
# Add --trim to the first command if TRIM is True above. Columns should agree to
# ~1e-12. If they do not, stop and find out why before refitting anything.

import subprocess, sys
repo = str(Path.home() / "nk92-ca-data")
cli = "/tmp/cli_ca.csv"
cmd = [sys.executable, f"{repo}/parse_ca_fcs.py", str(data_dir), "-o", cli,
       "--nbins", str(n_bins)] + (["--trim"] if TRIM else [])
print(" ".join(cmd)); print(subprocess.run(cmd, capture_output=True, text=True).stdout)
print(subprocess.run([sys.executable, f"{repo}/compare_ca_csv.py", str(OUT), cli],
                     capture_output=True, text=True).stdout)